# Pharmaceutical RAG — Working Demo

This demo uses the **same full RAG pipeline** as the main project, but first creates a small synthetic pharmaceutical document bundle so the entire workflow can be demonstrated immediately.

Demo flow: synthetic pharma PDF → page extraction → document classification/boundaries → metadata → chunks → MiniLM embeddings → vector + BM25 retrieval → query routing → reranking → grounded Gemini answer → page/source citations → Gradio UI.

In [ ]:
!pip install -q llama-index llama-index-llms-google-genai llama-index-embeddings-huggingface llama-index-retrievers-bm25 sentence-transformers pymupdf nest_asyncio gradio reportlab

## Setup
Add GOOGLE_API_KEY to Colab Secrets before running this cell.

In [ ]:
import os, re, fitz, nest_asyncio, gradio as gr
from google.colab import files, userdata
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import QueryBundle
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.postprocessor import SentenceTransformerRerank

nest_asyncio.apply()
GOOGLE_API_KEY=userdata.get("GOOGLE_API_KEY")
if not GOOGLE_API_KEY: raise ValueError("Add GOOGLE_API_KEY to Colab Secrets.")
os.environ["GOOGLE_API_KEY"]=GOOGLE_API_KEY
GEMINI_MODEL="gemini-3.5-flash"  # change only if your AI Studio account uses another available Gemini model
llm=GoogleGenAI(model=GEMINI_MODEL,api_key=GOOGLE_API_KEY)
print("Setup ready.")

## Create the demo pharmaceutical PDF and extract its pages
This replaces manual upload only for the demo. Everything after this uses the same RAG code.

In [ ]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter

pdf_name = "demo_pharmaceutical_bundle.pdf"
pdf_path = "/content/" + pdf_name

demo_pages = [
'''CERTIFICATE OF QUALITY\nProduct: Sterile Sample Kit\nMaterial Description: Sterile polypropylene sample collection kit\nLot Number: SK-2026-0915\nManufacture Date: 15 September 2026\nExpiration Date: 15 September 2028\nQuality Control Test Methods: Visual inspection, dimensional verification, sterility test\nGamma Irradiation: Passed, validated dose 25-40 kGy\nStorage Conditions: Store at 15-25 C in a dry area away from direct sunlight.\nStatus: PASS''',
'''CERTIFICATE OF QUALITY - CONTINUED\nProduct: Sterile Sample Kit\nLot Number: SK-2026-0915\nSterility Test: PASS\nVisual Inspection: PASS\nDimensional Verification: PASS\nApproved by: Quality Assurance\nApproval Date: 18 September 2026''',
'''PACKAGING SPECIFICATION\nProduct: Sterile Sample Kit\nPrimary Tray Part Number: PT-4401\nTyvek Lid Part Number: TL-2205\nCarton Part Number: CT-9010\nPackaging Configuration: 10 sterile kits per carton.\nChange Notice: Configuration changed from 8 kits per carton to 10 kits per carton effective 01 August 2026.''',
'''BSE/TSE DECLARATION\nProduct: Sterile Sample Kit\nSupplier: Example Medical Components Inc.\nDeclaration Date: 20 July 2026\nThe supplied polypropylene components contain no materials of animal origin.\nBSE/TSE Compliance Review Date: 20 July 2026\nStatus: Compliant based on supplier declaration.''',
'''SUPPLIER QUALIFICATION\nSupplier: Example Medical Components Inc.\nSupplier ID: EMC-1042\nQualification Status: Approved\nAudit Date: 12 June 2026\nNext Review Date: 12 June 2028\nApproved Material: Medical-grade polypropylene components.'''
]

c = canvas.Canvas(pdf_path, pagesize=letter)
width, height = letter
for page_text in demo_pages:
    text_obj = c.beginText(55, height - 70)
    text_obj.setFont("Helvetica", 10)
    for line in page_text.split("\n"):
        text_obj.textLine(line)
    c.drawText(text_obj)
    c.showPage()
c.save()

pdf=fitz.open(pdf_path)
pages=[{"page_number":i+1,"text":p.get_text("text").strip()} for i,p in enumerate(pdf)]
pdf.close()
print(f"Demo PDF created: {pdf_name}")
print("Pages:", len(pages))

## Page-level classification and document-boundary detection

In [ ]:
DOC_TYPES=["Cover Letter","Certificate of Quality","Packaging Specification","BSE/TSE Declaration","Material Description","Supplier Qualification","Chain of Custody","Other"]

def ask(prompt):
    return str(llm.complete(prompt)).strip()

def clean_type(x):
    n=re.sub(r"[^a-z0-9]+"," ",str(x).lower()).strip()
    for label in DOC_TYPES:
        l=re.sub(r"[^a-z0-9]+"," ",label.lower()).strip()
        if l in n or n in l: return label
    return "Other"

def classify(text):
    if not text: return "Other"
    return clean_type(ask("""Classify this pharmaceutical page into exactly one label:
Cover Letter
Certificate of Quality
Packaging Specification
BSE/TSE Declaration
Material Description
Supplier Qualification
Chain of Custody
Other
Return only the label.

PAGE:
"""+text[:12000]))

def same_document(prev,current,prev_type):
    if not prev or not current: return False
    prompt=f"""Decide whether these two consecutive pharmaceutical PDF pages belong to the SAME logical document.
Previous type: {prev_type}
Consider headings, tables, page numbering, product/lot identifiers, titles and signatures.
Return only Yes or No.

PREVIOUS:
{prev[-6000:]}

CURRENT:
{current[:6000:]}"""
    return ask(prompt).lower().startswith("yes")

page_meta=[]
doc_id=0
doc_type=None
for i,p in enumerate(pages):
    if i==0:
        doc_type=classify(p["text"])
    else:
        if not same_document(pages[i-1]["text"],p["text"],doc_type):
            doc_id+=1
            doc_type=classify(p["text"])
    page_meta.append({"document_id":doc_id,"page_number":p["page_number"],"doc_type":doc_type,"text":p["text"]})
    print(p["page_number"],doc_id,doc_type)

## Group logical documents and preserve metadata

In [ ]:
groups={}
for p in page_meta:
    d=p["document_id"]
    groups.setdefault(d,{"doc_type":p["doc_type"],"start":p["page_number"],"end":p["page_number"],"parts":[]})
    groups[d]["end"]=p["page_number"]
    groups[d]["parts"].append(p["text"])

documents=[]
for d,g in groups.items():
    documents.append(Document(
        text="\n\n".join(g["parts"]),
        metadata={"document_id":d,"doc_type":g["doc_type"],"page_start":g["start"],"page_end":g["end"],"source_file":pdf_name}
    ))
print("Logical documents:",len(documents))

## Chunk, embed and index
MiniLM runs locally. Chunks use 512 tokens with 50-token overlap.

In [ ]:
embed_model=HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.llm=llm
Settings.embed_model=embed_model
splitter=SentenceSplitter(chunk_size=512,chunk_overlap=50)
nodes=splitter.get_nodes_from_documents(documents)
for i,n in enumerate(nodes): n.metadata["chunk_index"]=i

index=VectorStoreIndex(nodes,embed_model=embed_model)
bm25=BM25Retriever.from_defaults(nodes=nodes,similarity_top_k=min(6,len(nodes)))
reranker=SentenceTransformerRerank(model="cross-encoder/ms-marco-MiniLM-L-6-v2",top_n=min(4,len(nodes)))
print("Chunks indexed:",len(nodes))

## Metadata-aware hybrid retrieval

In [ ]:
def route_type(query):
    labels="\n".join(DOC_TYPES)
    return clean_type(ask(f"""Route this question to exactly one pharmaceutical document type:
{labels}
Return only the label.
QUESTION: {query}"""))

def vector_retriever(doc_type=None,k=6):
    kw={"similarity_top_k":min(k,len(nodes))}
    if doc_type and doc_type!="Other":
        kw["filters"]=MetadataFilters(filters=[MetadataFilter(key="doc_type",value=doc_type,operator=FilterOperator.EQ)])
    return index.as_retriever(**kw)

def dedupe(items):
    out={}
    for x in items:
        nid=x.node.node_id
        if nid not in out or (x.score or -1e9)>(out[nid].score or -1e9): out[nid]=x
    return list(out.values())

def retrieve(query):
    routed=route_type(query)
    candidates=dedupe(
        vector_retriever(routed).retrieve(query)+
        vector_retriever().retrieve(query)+
        bm25.retrieve(query)
    )
    ranked=reranker.postprocess_nodes(candidates,query_bundle=QueryBundle(query_str=query)) if candidates else []
    return routed,ranked[:4]

## Grounded generation with page/source citations

In [ ]:
def answer_question(query):
    routed,hits=retrieve(query)
    if not hits:
        return "I could not find relevant evidence in the uploaded document.",routed,[]

    blocks=[]
    for i,h in enumerate(hits,1):
        m=h.metadata
        pages=str(m.get("page_start")) if m.get("page_start")==m.get("page_end") else f'{m.get("page_start")}-{m.get("page_end")}'
        blocks.append(f"[Source {i} | {m.get('doc_type')} | pages {pages}]\n{h.get_content()}")
    context="\n\n".join(blocks)

    prompt=f"""You are a pharmaceutical document QA assistant.
Answer ONLY from the evidence below.
Do not use outside facts or guess.
If the evidence is insufficient, say: The uploaded document does not provide enough information to answer this question.
Keep identifiers, dates, batch numbers, part numbers and test names exact.
Cite evidence as [Source 1], [Source 2], etc.
If sources conflict, state the conflict.

QUESTION:
{query}

EVIDENCE:
{context}

ANSWER:"""
    return ask(prompt),routed,hits

def show_answer(q):
    ans,routed,hits=answer_question(q)
    print("Routed to:",routed)
    print("\n",ans)
    print("\nSources:")
    for i,h in enumerate(hits,1):
        m=h.metadata
        print(i,m.get("doc_type"),f'pages {m.get("page_start")}-{m.get("page_end")}',m.get("source_file"))

## Test questions

In [ ]:
tests=[
    "What is the material description for this product?",
    "What are the part numbers listed in the packaging specification?",
    "Were there any packaging configuration changes?",
    "What is the BSE/TSE compliance review date?",
    "Who is the supplier and what is its qualification status?",
    "What test methods were used for quality control?",
    "Did this lot pass gamma irradiation testing?",
    "What are the storage conditions specified in the certificate?"
]
for q in tests:
    print("\n"+"="*80)
    print("QUESTION:",q)
    show_answer(q)

## Gradio interface

In [ ]:
def rag_ui(question):
    if not question.strip(): return "Please enter a question."
    ans,routed,hits=answer_question(question.strip())
    src=[]
    for i,h in enumerate(hits,1):
        m=h.metadata
        p=str(m.get("page_start")) if m.get("page_start")==m.get("page_end") else f'{m.get("page_start")}-{m.get("page_end")}'
        src.append(f"[Source {i}] {m.get('doc_type')} | page(s) {p} | {m.get('source_file')}")
    return ans+"\n\n---\nQuery routed to: "+routed+"\n\nSources:\n"+("\n".join(src) if src else "No sources retrieved.")

demo=gr.Interface(
    fn=rag_ui,
    inputs=gr.Textbox(lines=2,label="Ask about the uploaded pharmaceutical document"),
    outputs=gr.Textbox(lines=14,label="Grounded RAG answer"),
    title="Pharmaceutical Document RAG Assistant — Live Demo",
    description="Hybrid retrieval with metadata routing and page-level source information.",
    examples=[["What test methods were used for quality control?"],["What are the BSE/TSE compliance dates?"]]
)
demo.launch(share=True)

## Notes
This pipeline uses PyMuPDF for digital PDFs. Image-only/scanned PDFs need OCR before this RAG stage. Never commit your Gemini API key; keep it in Colab Secrets.